# 06 · Newsvendor buffer (Phase 6) — headline output

| Newsvendor | Our project |
|---|---|
| Order quantity *Q* | buffer *b* added to the quoted COE price |
| Uncertain demand *D* | premium rise *R* between quote and purchase |
| Underage cost *c_u* | *R* > *b*: dealer absorbs the shortfall, *c_u* = S$1 per S$1 short |
| Overage cost *c_o* | *b* > *R*: car overpriced → lost or discounted sales; not public |
| Critical ratio *c_u/(c_u+c_o)* | percentile τ of the forecast distribution to charge |

$$b^* = \max\bigl(0,\; F_R^{-1}(\tau^*)\bigr), \qquad \tau^* = \frac{k}{1+k}, \quad k = c_u/c_o$$

Because *c_o* is unknown we sweep τ = 0.5 … 0.95 (k = 1 … 19, "underpricing costs k times as much as overpricing").

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import *
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [2]:
from src.newsvendor import tau_to_k, implied_cost_ratios, condition_cutoffs, buffer_table, buffer_sgd, assign_condition
fc = pd.read_csv(PROC / "forecasts.csv")
feat = pd.read_csv(PROC / "coe_features.csv")
best = json.load(open(PROC / "best_model.json"))["best_model"]
print("Buffer model:", best)
pd.DataFrame({"tau": TAUS, "k = c_u / c_o": [round(tau_to_k(t), 1) for t in TAUS],
              "meaning": [f"underpricing costs {tau_to_k(t):.1f}x overpricing" for t in TAUS]})

Buffer model: M6


,tau,k = c_u / c_o,meaning
0,0.500,1.000,underpricing costs 1.0x overpricing
1,0.600,1.500,underpricing costs 1.5x overpricing
2,0.700,2.300,underpricing costs 2.3x overpricing
3,0.800,4.000,underpricing costs 4.0x overpricing
4,0.900,9.000,underpricing costs 9.0x overpricing
5,0.950,19.000,underpricing costs 19.0x overpricing


## Cost ratios implied by what dealers do today (h = 2, 2014–2026)

In [3]:
base = fc[(fc.model == "M0") & (fc.h == H) & (fc.tau == 0.5)].dropna(subset=["actual_sgd"])
imp = implied_cost_ratios(base)
display(imp.round(2))
imp_p = pd.concat([implied_cost_ratios(g).assign(period=p) for p, g in base.groupby("period")])
display(imp_p[imp_p.category == "All"].pivot(index="rule", columns="period", values="implied_tau").round(2))
imp.to_csv(PROC / "implied_cost_ratios.csv", index=False)
a = imp[imp.category == "All"].set_index("rule")
print(f"No buffer covers {a.loc['R0 No buffer','implied_tau']:.0%} of sales -> acts as if k = {a.loc['R0 No buffer','implied_k']:.2f} "
      f"(overpricing treated as the bigger risk).")
print(f"Fixed 5% covers {a.loc['R1 Fixed 5%','implied_tau']:.0%} -> acts as if k = {a.loc['R1 Fixed 5%','implied_k']:.2f}.")

,category,rule,implied_tau,implied_k
0,All,R0 No buffer,0.460,0.840
1,All,R1 Fixed 5%,0.720,2.600
2,A,R0 No buffer,0.450,0.820
3,A,R1 Fixed 5%,0.740,2.820
4,B,R0 No buffer,0.460,0.860
5,B,R1 Fixed 5%,0.700,2.390
6,E,R0 No buffer,0.450,0.830
7,E,R1 Fixed 5%,0.720,2.630


period,2014-19,2020-22,2023-26
rule,,,
R0 No buffer,0.550,0.380,0.360
R1 Fixed 5%,0.780,0.590,0.720


No buffer covers 46% of sales -> acts as if k = 0.84 (overpricing treated as the bigger risk).
Fixed 5% covers 72% -> acts as if k = 2.60.


## The dealer's buffer table

Rows: category × market condition = **recent volatility** at the quote round (`vol_12`, the std. dev. of 1-round % changes over the last 12 rounds; terciles per category). Volatility is used rather than the bid ratio because the EDA and the driver regression found that the bid ratio barely predicts the next rise, while volatility clusters (calm and turbulent spells persist). The bid-ratio version is kept below for comparison. Columns: cost ratio *k*. Each cell: the median newsvendor buffer from the rounds in that cell (2014–2026, out-of-sample) as % of the premium, applied to today's premium (Oct 2026 round 2). The backtest columns use the actual per-round buffers.

In [4]:
cuts = condition_cutoffs(feat, "vol_12")
latest = feat.sort_values("t").groupby("category").premium.last().to_dict()
print("volatility tercile cut-offs (std. dev. of 1-round changes):", {c: tuple(f"{x:.1%}" for x in v) for c, v in cuts.items()})
print("latest premiums:", latest)
fb = fc[(fc.model == best) & (fc.h == H)].merge(feat[["category", "t", "vol_12", "vol_ewma"]], on=["category", "t"])
bt_table = buffer_table(fb, cuts, latest, "vol_12")
bt_table.to_csv(PROC / "buffer_table.csv", index=False)
cell = bt_table.assign(cell=lambda d: d.apply(lambda r: f"S${r.buffer_sgd:,.0f} ({r.buffer_pct:.1f}%)", axis=1))
pretty = cell.pivot_table(index=["category", "condition", "signal_range"], columns="k", values="cell", aggfunc="first")
pretty = pretty.reindex(sorted(pretty.index, key=lambda i: (i[0], {"Low": 0, "Medium": 1, "High": 2}[i[1]])))
pretty.columns = [f"k={c}" for c in pretty.columns]
pretty.to_csv(PROC / "buffer_table_pretty.csv")
pretty

volatility tercile cut-offs (std. dev. of 1-round changes): {'A': ('4.7%', '7.1%'), 'B': ('4.9%', '7.1%'), 'E': ('4.1%', '5.5%')}
latest premiums: {'A': 132004, 'B': 131901, 'E': 135790}


k=1.0           k=1.5           k=2.3            k=4.0             k=9.0            k=19.0
category condition signal_range                                                                                                   
A        Low       <= 4.7%       S$677 (0.5%)  S$2,463 (1.9%)  S$4,915 (3.7%)   S$7,984 (6.0%)   S$11,927 (9.0%)  S$15,097 (11.4%)
         Medium    4.7%-7.1%     S$710 (0.5%)  S$2,763 (2.1%)  S$5,449 (4.1%)   S$8,765 (6.6%)  S$13,156 (10.0%)  S$16,783 (12.7%)
         High      > 7.1%        S$322 (0.2%)  S$3,419 (2.6%)  S$7,140 (5.4%)  S$11,878 (9.0%)  S$19,159 (14.5%)  S$24,380 (18.5%)
B        Low       <= 4.9%       S$761 (0.6%)  S$2,632 (2.0%)  S$5,179 (3.9%)   S$8,395 (6.4%)   S$12,465 (9.5%)  S$16,146 (12.2%)
         Medium    4.9%-7.1%     S$707 (0.5%)  S$2,916 (2.2%)  S$5,889 (4.5%)   S$9,609 (7.3%)  S$14,871 (11.3%)  S$19,334 (14.7%)
         High      > 7.1%        S$810 (0.6%)  S$3,191 (2.4%)  S$6,464 (4.9%)  S$10,314 (7.8%)  S$16,616 (12.6%)  S$21,682 (16.4%)
E        Low       <= 4.1%       S$577 (0.4%)  S$2,180 (1.6%)  S$4,361 (3.2%)   S$7,055 (5.2%)   S$10,695 (7.9%)  S$13,813 (10.2%)
         Medium    4.1%-5.5%     S$641 (0.5%)  S$2,338 (1.7%)  S$4,560 (3.4%)   S$7,448 (5.5%)   S$11,529 (8.5%)  S$14,803 (10.9%)
         High      > 5.5%        S$339 (0.2%)  S$2,634 (1.9%)  S$5,508 (4.1%)   S$9,222 (6.8%)  S$14,585 (10.7%)  S$19,124 (14.1%)

In [5]:
print("Share of sales underwater with that buffer (2014-2026 backtest), %:")
uw = bt_table.pivot_table(index=["category", "condition"], columns="k", values="underwater_pct")
display(uw.reindex(sorted(uw.index, key=lambda i: (i[0], {"Low": 0, "Medium": 1, "High": 2}[i[1]]))).round(1))

Share of sales underwater with that buffer (2014-2026 backtest), %:


k                   1.000   1.500   2.300   4.000   9.000   19.000
category condition                                                
A        Low        52.100  44.600  32.200  18.200   9.900   5.000
         Medium     57.600  42.400  31.500  17.400   8.700   3.300
         High       45.900  37.600  24.700  12.900   9.400   3.500
B        Low        53.200  43.700  35.700  21.400  13.500   7.100
         Medium     44.000  34.000  21.000  15.000   6.000   4.000
         High       55.600  48.600  34.700  26.400  12.500   5.600
E        Low        58.600  50.500  46.800  36.900  22.500  15.300
         Medium     43.900  37.700  28.900  15.800   7.000   3.500
         High       46.600  32.900  26.000  16.400   8.200   4.100

In [6]:
r = bt_table[(bt_table.category == "B") & (bt_table.condition == "High") & (bt_table.tau == 0.8)].iloc[0]
c = bt_table[(bt_table.category == "B") & (bt_table.condition == "Low") & (bt_table.tau == 0.8)].iloc[0]
guidance = (f"A dealer who rates underpricing four times as costly as overpricing (k = 4, τ = 0.8) should add about "
            f"{r.buffer_pct:.1f}% (S${r.buffer_sgd:,.0f} at today's premium) to a Cat B package after a turbulent spell "
            f"(1-round changes over the last 12 rounds with a std. dev. above {cuts['B'][1]:.1%}), against {c.buffer_pct:.1f}% "
            f"(S${c.buffer_sgd:,.0f}) in a calm market (below {cuts['B'][0]:.1%}); in the 2014-2026 backtest those buffers "
            f"were too small on {r.underwater_pct:.0f}% and {c.underwater_pct:.0f}% of such sales.")
print(guidance)
_ = (REPORT / "buffer_guidance.txt").write_text(guidance)

A dealer who rates underpricing four times as costly as overpricing (k = 4, τ = 0.8) should add about 7.8% (S$10,314 at today's premium) to a Cat B package after a turbulent spell (1-round changes over the last 12 rounds with a std. dev. above 7.1%), against 6.4% (S$8,395) in a calm market (below 4.9%); in the 2014-2026 backtest those buffers were too small on 26% and 21% of such sales.


## Recent regime (2023–2026 only)

Premiums and volatility are higher since 2023, so the same τ needs a bigger buffer. A dealer pricing today should look at this version too.

In [7]:
bt_recent = buffer_table(fb[fb.period == "2023-26"], cuts, latest)
bt_recent.to_csv(PROC / "buffer_table_2023_26.csv", index=False)
rp = bt_recent.pivot_table(index=["category", "condition"], columns="k", values="buffer_pct")
rp.reindex(sorted(rp.index, key=lambda i: (i[0], {"Low": 0, "Medium": 1, "High": 2}[i[1]]))).round(1)

k                   1.000   1.500   2.300   4.000   9.000   19.000
category condition                                                
A        Low         0.700   2.000   3.800   6.200   9.200  11.800
         Medium      0.600   2.000   4.100   6.600   9.800  12.500
         High        0.900   3.300   6.800  10.800  15.700  20.000
B        Low         0.800   2.200   4.100   6.700  10.000  12.800
         Medium      0.800   2.500   4.700   7.600  11.500  14.600
         High        1.100   3.800   7.800  12.500  18.300  23.100
E        Low         0.500   1.600   3.200   5.200   7.700   9.800
         Medium      0.600   1.800   3.500   5.800   8.600  11.000
         High        0.700   2.600   5.300   8.500  12.300  15.600

## Which market signal moves the buffer? Volatility vs the bid ratio

The same table cut by bid-ratio terciles (the original plan's condition). The spread is the largest minus the smallest buffer across the three conditions, in % points of the premium: a signal that does not change the buffer is no use to a dealer.

In [8]:
bcuts = condition_cutoffs(feat, "bid_ratio")
bt_bid = buffer_table(fb, bcuts, latest, "bid_ratio")
bt_bid.to_csv(PROC / "buffer_table_by_bidratio.csv", index=False)
print("Buffer % of premium by bid-ratio condition:")
display(bt_bid.pivot_table(index=["category", "condition"], columns="k", values="buffer_pct").round(1))
def spread(t):
    p = t.pivot_table(index=["category", "k"], columns="condition", values="buffer_pct")
    return p.max(axis=1) - p.min(axis=1)
comp = pd.DataFrame({"spread by volatility (pp)": spread(bt_table), "spread by bid ratio (pp)": spread(bt_bid)})
comp_k = comp.reset_index().pivot(index="k", columns="category")
comp_k.round(1)

Buffer % of premium by bid-ratio condition:


k                   1.000   1.500   2.300   4.000   9.000   19.000
category condition                                                
A        High        0.700   2.200   4.300   7.100  10.700  13.800
         Low         0.400   2.000   4.000   6.600  10.000  12.700
         Medium      0.500   2.100   4.100   6.800  10.400  13.400
B        High        0.800   2.400   4.700   7.700  11.800  15.300
         Low         0.400   2.000   4.100   6.600  10.700  13.900
         Medium      0.600   2.200   4.400   7.200  11.100  14.400
E        High        0.500   1.800   3.500   5.600   8.600  11.100
         Low         0.300   1.700   3.400   5.500   8.700  11.200
         Medium      0.400   1.800   3.400   5.400   8.700  11.200

spread by volatility (pp)             spread by bid ratio (pp)            
category                         A     B     E                        A     B     E
k                                                                                  
1.000                        0.300 0.100 0.200                    0.300 0.300 0.200
1.500                        0.700 0.400 0.300                    0.200 0.400 0.100
2.300                        1.700 1.000 0.800                    0.300 0.600 0.100
4.000                        2.900 1.500 1.600                    0.500 1.100 0.200
9.000                        5.500 3.100 2.900                    0.700 1.100 0.100
19.000                       7.000 4.200 3.900                    1.100 1.500 0.100

In [9]:
sign = bt_table.pivot_table(index=["category", "k"], columns="condition", values="buffer_pct")
mono = ((sign["Low"] <= sign["Medium"]) & (sign["Medium"] <= sign["High"])).groupby("category").mean()
sign_b = bt_bid.pivot_table(index=["category", "k"], columns="condition", values="buffer_pct")
mono_b = ((sign_b["Low"] <= sign_b["Medium"]) & (sign_b["Medium"] <= sign_b["High"])).groupby("category").mean()
print("Share of k values where the buffer rises Low <= Medium <= High:")
print(pd.DataFrame({"volatility": mono, "bid ratio": mono_b}).round(2).to_string())
print(f"\nMean spread across conditions: volatility {comp.iloc[:, 0].mean():.1f} pp vs bid ratio {comp.iloc[:, 1].mean():.1f} pp")

Share of k values where the buffer rises Low <= Medium <= High:
          volatility  bid ratio
category                       
A              0.830      1.000
B              0.830      1.000
E              0.830      0.330

Mean spread across conditions: volatility 2.1 pp vs bid ratio 0.5 pp


## Today's recommendation (forecast made at the last round, October 2026 round 2, for a purchase two rounds later)

In [10]:
last = fb[fb.round_id == fb.round_id.max()].copy()
last["buffer_sgd"] = buffer_sgd(last.q_pct, last.premium_now)
last["k"] = last.tau.map(lambda t: round(tau_to_k(t), 1))
last["condition"] = assign_condition(last, cuts, "vol_12")
now = last.pivot_table(index=["category", "premium_now", "vol_12", "vol_ewma", "bid_ratio", "condition"], columns="k", values="buffer_sgd")
now.columns = [f"k={c}" for c in now.columns]
now.to_csv(PROC / "buffer_today.csv")
now.round(0)

,,,,,,k=1.0,k=1.5,k=2.3,k=4.0,k=9.0,k=19.0
category,premium_now,vol_12,vol_ewma,bid_ratio,condition,,,,,,
A,132004,0.023,0.038,1.322,Low,778.000,"2,070.000","3,870.000","6,101.000","9,101.000","11,876.000"
B,131901,0.028,0.046,1.245,Low,928.000,"2,471.000","4,621.000","7,284.000","10,865.000","14,179.000"
E,135790,0.013,0.029,1.522,Low,612.000,"1,629.000","3,046.000","4,802.000","7,163.000","9,347.000"


## The dealer's one-line formula (volatility-scaled rule)

M6 reduces to a rule a dealer can run in a spreadsheet:

$$\text{buffer} = z_k \times \text{recent volatility} \times \text{today's premium}$$

where recent volatility is the EWMA of squared 1-round % changes (λ = 0.94) and *z_k* is read off the table below (the τ* = k/(1+k) quantile of every past 2-round rise divided by the volatility at its quote). It is as simple as "add 5%", but the percentage grows when the market has been turbulent and shrinks when it has been calm.

In [11]:
from src.models import z_multipliers
z = z_multipliers(feat, H)
lastf = feat.sort_values("t").groupby("category").tail(1).set_index("category")
form = pd.DataFrame({"tau": TAUS, "k": [round(tau_to_k(t), 1) for t in TAUS], "z": z.values})
for c in CATEGORIES:
    form[f"Cat {c} buffer %"] = np.maximum(0, form.z * lastf.loc[c, "vol_ewma"]) * 100
    form[f"Cat {c} buffer S$"] = form[f"Cat {c} buffer %"] / 100 * lastf.loc[c, "premium"]
form.to_csv(PROC / "buffer_formula.csv", index=False)
print("today's EWMA volatility:", {c: f"{lastf.loc[c, 'vol_ewma']:.1%}" for c in CATEGORIES},
      "| long-run median:", {c: f"{v:.1%}" for c, v in feat.groupby("category").vol_ewma.median().items()})
form.round(2)

today's EWMA volatility: {'A': '3.8%', 'B': '4.6%', 'E': '2.9%'} | long-run median: {'A': '6.4%', 'B': '6.7%', 'E': '5.3%'}


,tau,k,z,Cat A buffer %,Cat A buffer S$,Cat B buffer %,Cat B buffer S$,Cat E buffer %,Cat E buffer S$
0,0.500,1.000,0.150,0.590,777.600,0.700,928.400,0.450,612.020
1,0.600,1.500,0.410,1.570,"2,069.610",1.870,"2,470.940",1.200,"1,628.910"
2,0.700,2.300,0.760,2.930,"3,870.320",3.500,"4,620.850",2.240,"3,046.180"
3,0.800,4.000,1.200,4.620,"6,100.590",5.520,"7,283.610",3.540,"4,801.540"
4,0.900,9.000,1.790,6.890,"9,100.660",8.240,"10,865.460",5.270,"7,162.790"
5,0.950,19.000,2.340,9.000,"11,875.680",10.750,"14,178.600",6.880,"9,346.900"


In [12]:
print("Example: z at k = 4 is", f"{z.loc[0.8]:.2f}", "so the buffer is", f"{z.loc[0.8]:.2f}", "x volatility:")
for v in [0.03, 0.05, 0.08, 0.12]:
    print(f"  volatility {v:.0%} -> buffer {z.loc[0.8] * v:.1%} of the premium")
print("For comparison, fixed 5% equals the k = 4 formula only when volatility is", f"{0.05 / z.loc[0.8]:.1%}")

Example: z at k = 4 is 1.20 so the buffer is 1.20 x volatility:
  volatility 3% -> buffer 3.6% of the premium
  volatility 5% -> buffer 6.0% of the premium
  volatility 8% -> buffer 9.6% of the premium
  volatility 12% -> buffer 14.4% of the premium
For comparison, fixed 5% equals the k = 4 formula only when volatility is 4.2%


## Today's buffer under each candidate model (k = 4 and k = 9)

The models are statistically tied (Diebold–Mariano, notebook 05), so it is worth seeing how much today's recommendation depends on the choice.

In [13]:
tod = fc[(fc.h == H) & (fc.round_id == fc.round_id.max()) & fc.model.isin(["M1", "M3", "M5", "M6", "M7"]) & fc.tau.isin([0.8, 0.9])].copy()
tod["buffer_sgd"] = buffer_sgd(tod.q_pct, tod.premium_now)
tod["k"] = tod.tau.map(lambda t: round(tau_to_k(t), 1))
cmp_today = tod.pivot_table(index=["category", "k"], columns="model", values="buffer_sgd")
cmp_today.to_csv(PROC / "buffer_today_by_model.csv")
cmp_today.round(0)

model                  M1         M3         M5         M6         M7
category k                                                           
A        4.000  7,127.000  8,662.000  7,895.000  6,101.000  7,589.000
         9.000 10,380.000 10,716.000 10,548.000  9,101.000 10,446.000
B        4.000  7,617.000 10,016.000  8,816.000  7,284.000  7,089.000
         9.000  9,548.000 12,780.000 11,164.000 10,865.000  9,944.000
E        4.000  6,246.000  8,695.000  7,470.000  4,802.000  5,023.000
         9.000  7,711.000 11,833.000  9,772.000  7,163.000  6,761.000

## Optional: scenarios for the overage cost *c_o* (labelled assumptions, not estimates)

$c_o \approx$ (chance of losing the sale per extra S$1,000) × (dealer margin per car) ÷ 1,000. With *c_u* = 1, k = 1/c_o.

In [14]:
sc = pd.DataFrame({"scenario": ["Low c_o (loyal buyers)", "Medium", "High c_o (price-sensitive buyers)"],
                   "p_lose_per_1000": [0.02, 0.05, 0.10], "margin_per_car": [5000, 8000, 10000]})
sc["c_o"] = sc.p_lose_per_1000 * sc.margin_per_car / 1000
sc["k"] = 1 / sc.c_o; sc["tau_star"] = sc.k / (1 + sc.k)
sc["nearest_tau"] = sc.tau_star.map(lambda t: min(TAUS, key=lambda x: abs(x - t)))
sc

,scenario,p_lose_per_1000,margin_per_car,c_o,k,tau_star,nearest_tau
0,Low c_o (loyal buyers),0.020,5000,0.100,10.000,0.909,0.900
1,Medium,0.050,8000,0.400,2.500,0.714,0.700
2,High c_o (price-sensitive buyers),0.100,10000,1.000,1.000,0.500,0.500
